# 第 1 周第 2 天作业 —— 网页抓取 + 购机顾问

## 练习目标（理念）

用课程 Day 2 的思路：先 **scrape（抓取）** 网页正文，再把正文塞进 **Chat Completions** 的 user 消息，让模型当「购机顾问」做摘要与建议。

- **输入**：若干 Apple 官网 iPhone 产品页 URL
- **处理**：`fetch_website_contents` 抓正文 → 拼进 prompt
- **对比**：同一套 `homework_day2_message` 分别打给本地 `llama3.2` 与云端 `gpt-5-nano`

## 和本课概念的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| 网页抓取 | `from scraper import fetch_website_contents` |
| system / user messages | 顾问角色 +「学生需求 + 抓取正文」 |
| Ollama（OpenAI 兼容） | `base_url=http://localhost:11434/v1` |
| 云端 OpenAI | `OpenAI()` + `gpt-5-nano` |

## 怎么跑

1. 确保能 import 到仓库里的 `week1/scraper.py`（本笔记本用相对路径往上两级再进 `week1`）
2. 本地试跑前：`ollama pull llama3.2` 并启动 Ollama；云端试跑前在 `.env` 配好 `OPENAI_API_KEY`
3. 依次跑「General setup」与两次 Try，对比两家模型的建议质量


## 作者说明（怎么用这份作业）

配置好环境后，直接跑下面两次尝试（Try 1 / Try 2），观察两个模型在「帮你选新 iPhone」这件事上的表现差异。

作者体感：ChatGPT 一侧明显更强。

补充：作者是德语使用者，因此输出与网站默认是德语版 Apple 站（`/de/`）。若要换语言，把下面的 URL 改成对应地区路径即可。


### 通用准备（导入 + 拼 messages）


In [ ]:
# ========== 导入：抓取工具 + OpenAI SDK + 笔记本展示 ==========

# 导入标准库 sys：后面用来改模块搜索路径（sys.path）
import sys
# 导入标准库 os：拼路径用
import os
# 把仓库 week1 目录加进 sys.path，才能 import 到课程自带的 scraper
sys.path.append(os.path.join('..', '..', 'week1'))
# 从 scraper 导入 fetch_website_contents：抓取网页正文（Day 2 常用工具）
from scraper import fetch_website_contents
# 从 operator 导入 concat：本格导入了但后面未使用（保持原逻辑不动）
from operator import concat
# 从 openai 导入 OpenAI 客户端：云端与 Ollama 兼容端点都能用
from openai import OpenAI
# 从 IPython.display 导入 Markdown / display：在笔记本里漂亮渲染模型回答
from IPython.display import Markdown, display
# 从 dotenv 导入 load_dotenv：后面云端调用前再加载 .env
from dotenv import load_dotenv


In [ ]:
# ========== 提示词 + 抓取多个产品页 + 组装 messages ==========

# system prompt：定「购机顾问」角色与约束——只依据用户提供的抓取事实（保留英文，改译会改变行为）
system_prompt_homework_day2 = """
You are tech advisor that knows about how newest mobiles like the iPhone work and what different user groups need, 
like best cameras for Pro users or battery life for average consumers. 
You advise fact-based and thorough on the best deals for each of your customers.
You deal with scraped websites given to you by ignoring random text from headers and footers of websites 
and focusing on the content that actually presents products. 
IMPORTANT: Only use facts that you got by the user via prompting.
Try hard to really identify which text is noise and which makes the product descriptions.
"""
# user 前缀：说明学生场景（仍用 iPhone 13 Pro）并要求先摘要再给建议（英文 prompt 不翻译）
user_prompt_prefix_homework_day2 = """
Hi, as a student that still has the iPhone 13 Pro but wants a newer iPhone, 
i need some advice on which iPhone to buy. Can you help me choose from apples website? 
Here are the contents of apple's iphone websites that i obtained using a scraper.
Provide a short summary of the text from this website-scraper and afterwards help me choose the best option. 
If it includes news or announcements, then summarize these too. 
"""

# 三个德语区 Apple 产品页 URL（字符串影响抓取目标，保持原样）
url_1 = "https://www.apple.com/de/iphone-17-pro/"
url_2 = "https://www.apple.com/de/iphone-air/"
url_3 = "https://www.apple.com/de/iphone-17/"

# 把三个 URL 放进列表，方便 for 循环依次抓取
urls = [url_1, url_2, url_3]
# 累积所有页面抓取到的正文
website_content_homework_day2 = ""

# 逐个 URL 抓取，并把正文追加到同一个大字符串
for url in urls:
    website_content_homework_day2 += fetch_website_contents(url)

# messages：system 定顾问角色；user = 前缀说明 + 全部抓取正文
homework_day2_message = [
    {"role":"system", "content":system_prompt_homework_day2},
    {"role":"user", "content": user_prompt_prefix_homework_day2 + website_content_homework_day2}
]


### 尝试 1：本地 Ollama（llama3.2）


In [ ]:
# ========== 本地 Ollama：先 pull，再走 OpenAI 兼容端点提问 ==========

# Jupyter shell magic：在 shell 里执行 ollama pull，确保本机有 llama3.2
!ollama pull llama3.2

# Ollama 的 OpenAI 兼容 API 根地址
OLLAMA_BASE_URL = "http://localhost:11434/v1"
# 创建指向本地的客户端；api_key 对 Ollama 通常只是占位
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

# 非流式调用：把前面组装好的 homework_day2_message 发给本地模型
response = ollama.chat.completions.create(model="llama3.2", messages=homework_day2_message,)
# 用 Markdown 在笔记本里渲染完整回答
display(Markdown(response.choices[0].message.content))


### 尝试 2：云端 ChatGPT（gpt-5-nano）


In [ ]:
# ========== 云端 OpenAI：加载密钥后用同一套 messages 对比 ==========

# 加载 .env；override=True 用文件值覆盖进程里已有同名变量
load_dotenv(override=True)

# 云端客户端：默认读环境变量 OPENAI_API_KEY
openai = OpenAI()
# 模型 id 保持原样（gpt-5-nano）；messages 与本地试跑完全相同，便于公平对比
response = openai.chat.completions.create(model="gpt-5-nano", messages=homework_day2_message,)
# 同样用 Markdown 渲染回答
display(Markdown(response.choices[0].message.content))
